In [1]:
import numpy as np
from numba import njit

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import scipy.sparse as sp
from scipy.spatial.distance import directed_hausdorff

from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error

import optuna
import optuna.visualization as vis
from optuna.importance import PedAnovaImportanceEvaluator

import warnings

In [2]:
steps = 20000

tau_steps = 1

transient_steps_henon = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_henon + transient_steps_reservoir + tau_steps
total_steps_after_henon = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

# Init

In [3]:
henon_dataset = np.zeros((total_steps, 2))

rng = np.random.default_rng(42)
henon_dataset[0] = rng.random(2)

a = 1.4
b = 0.3

In [4]:
@njit
def henon_numba(steps, a=1.4, b=0.3, x0=0.0, y0=0.0):
    X = np.zeros(steps)
    Y = np.zeros(steps)
    X[0] = x0
    Y[0] = y0

    for i in range(1, steps):
        X[i] = 1 - a * X[i - 1] ** 2 + Y[i - 1]
        Y[i] = b * X[i - 1]

    return X, Y

In [5]:
henon_data_x, henon_data_y = henon_numba(total_steps)

henon_dataset = np.column_stack((henon_data_x, henon_data_y))
henon_dataset = henon_dataset[transient_steps_henon:]

In [6]:
henon_scaler = StandardScaler()
henon_scaled = henon_scaler.fit_transform(henon_dataset)

In [7]:
def henon_plot(data_list, names=None):
    fig = go.Figure()
    colors = ["white", "magenta"]

    for i, data in enumerate(data_list):
        fig.add_trace(
            go.Scatter(
                x=data[:, 0],
                y=data[:, 1],
                mode="markers",
                name=names[i] if names else f"Dataset {i+1}",
                marker=dict(color=colors[i % len(colors)], size=1),
            )
        )

    fig.update_layout(
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white"),
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
    )

    return fig

In [8]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(go.Scatter(
            x=actual, y=predicted, mode="markers",
            name="Data", marker=dict(color="rgba(50, 50, 200, 0.5)", size=5)
        ), row=1, col=col)

        min_val, max_val = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
        fig.add_trace(go.Scatter(
            x=[min_val, max_val], y=[min_val, max_val], mode="lines", 
            name="Ideal", line=dict(color="firebrick", dash="dash")
        ), row=1, col=col)

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

# Bayesian Init

In [9]:
@njit(fastmath=True, cache=True)
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha, noise):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = (henon_inputs + noise) @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

In [10]:
@njit(fastmath=True, cache=True)
def compute_closed_states(
    steps_start,
    steps_end,
    Y_pred_scaled,
    X_pred,
    W_in,
    W_res,
    bias,
    alpha,
    W_out,
    W_bias,
    res_size,
    scaler_mean,
    scaler_std,
):
    for i in range(steps_start, steps_end):
        u = Y_pred_scaled[i - 2]
        prev_state = X_pred[i - 1, :res_size]
        new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
        X_pred[i, :res_size] = (1.0 - alpha) * prev_state + alpha * new_state
        X_scaled = (X_pred[i, :res_size] - scaler_mean) / scaler_std
        Y_pred_scaled[i] = W_out @ X_scaled + W_bias
    return Y_pred_scaled

In [11]:
@njit(fastmath=True, cache=True)
def lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon):
    Q_test = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_test = np.zeros(2)
    for i in range(len(Y_test)):
        J = np.array([[-2.0 * a * Y_test[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_test)
        Q_test_raw, R = np.linalg.qr(Z)
        Q_test = np.ascontiguousarray(Q_test_raw)
        lyapunov_sums_test += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_test = lyapunov_sums_test / total_steps_after_henon

    Q_pred = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_pred = np.zeros(2)
    for i in range(len(Y_pred)):
        J = np.array([[-2.0 * a * Y_pred[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_pred)
        Q_pred_raw, R = np.linalg.qr(Z)
        Q_pred = np.ascontiguousarray(Q_pred_raw)
        lyapunov_sums_pred += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_pred = lyapunov_sums_pred / total_steps_after_henon

    le_loss = (lyapunov_exponent_test - lyapunov_exponent_pred) ** 2
    return le_loss[0]

In [12]:
def henon_closed(
    in_size,
    out_size,
    res_size,
    sparsity,
    spec_rad,
    alpha,
    input_scaling,
    bias_scaling,
    ridge_alpha,
    noise_val,
    tau_steps,
    trial=None
):
    rng = np.random.default_rng(42)
    bias = rng.uniform(-bias_scaling, bias_scaling, res_size)
    W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
    W_res_sparse = sp.random(
        res_size,
        res_size,
        density=sparsity,
        format="csr",
        random_state=rng,
        data_rvs=lambda l: rng.uniform(-1.0, 1.0, l),
    )
    try:
        eigenvalues, _ = sp.linalg.eigs(W_res_sparse, k=1, which="LM", ncv=20)
    except Exception as e:
        raise optuna.exceptions.TrialPruned()
    largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
    W_res = W_res_sparse * (spec_rad / largest_eigenvalue)
    W_res_dense = W_res.toarray()

    rng = np.random.default_rng(42)
    noise = rng.normal(
        0, noise_val, size=(len(henon_scaled) - test_steps - tau_steps, out_size)
    )
    X = compute_states(
        steps + transient_steps_reservoir - test_steps,
        henon_scaled[: -test_steps - tau_steps],
        res_size,
        W_in,
        W_res_dense,
        bias,
        alpha,
        noise,
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[transient_steps_reservoir:])
    Y_train = henon_scaled[transient_steps_reservoir + tau_steps : -test_steps]
    scaler_mean = scaler.mean_
    scaler_std = scaler.scale_

    with warnings.catch_warnings():
        warnings.filterwarnings("error", category=Warning)
        try:
            model = Ridge(alpha=ridge_alpha, solver="auto")
            model.fit(X_train, Y_train)
        except (Warning, ValueError):
            try:
                model = Ridge(alpha=ridge_alpha, solver="svd")
                model.fit(X_train, Y_train)
            except Exception:
                raise optuna.exceptions.TrialPruned()

    W_out = model.coef_
    W_bias = model.intercept_

    X_pred = np.zeros((test_steps, res_size))
    Y_pred_scaled = np.zeros((test_steps, out_size))
    Y_test = henon_dataset[-test_steps:]

    u = Y_train[-2]
    prev_state = X_train[-1]
    new_state = np.tanh(W_in @ u + W_res_dense @ prev_state + bias)
    X_pred[0] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[0].reshape(1, -1))
    Y_pred_scaled[0] = model.predict(X_pred_scaled)

    u = Y_train[-1]
    prev_state = X_pred[0]
    new_state = np.tanh(W_in @ u + W_res_dense @ prev_state + bias)
    X_pred[1] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[1].reshape(1, -1))
    Y_pred_scaled[1] = model.predict(X_pred_scaled)

    chunks = 16
    steps_per_chunk = test_steps // chunks
    for i in range(chunks):
        start_step = 2 + (steps_per_chunk * i)
        end_step = min(2 + (steps_per_chunk * (i + 1)), test_steps)
        compute_closed_states(
            start_step,
            end_step,
            Y_pred_scaled,
            X_pred,
            W_in,
            W_res_dense,
            bias,
            alpha,
            W_out,
            W_bias,
            res_size,
            scaler_mean,
            scaler_std,
        )

        loss = lyapunov_loss(
            Y_test,
            henon_scaler.inverse_transform(Y_pred_scaled),
            a,
            b,
            total_steps_after_henon,
        )

        if trial:
            trial.report(loss, 2 + steps_per_chunk * (i + 1))
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)

    return Y_test, Y_pred

# Bayesian MSE

In [161]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 10, 2000),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 0.1, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 0.1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )
    mse = root_mean_squared_error(Y_test, Y_pred)

    return mse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #24...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 9.192194761729169e-17.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


[Optuna] Processing Trial #99...

In [162]:
print(f"MSE={study.best_value}")
print(f"Params: {study.best_params}")

MSE=0.471350228314025
Params: {'res_size': 106, 'sparsity': 0.47559310737707644, 'spec_rad': 1.9193837150450144, 'alpha': 0.5624283664448213, 'input_scaling': 0.26896971075091153, 'bias_scaling': 0.10909174145812386, 'ridge_alpha': 0.0009084883621240411, 'noise_val': 0.01843062810647737}


In [163]:
params = study.best_trials[0].params
Y_test, Y_pred = henon_closed(
    in_size=2,
    out_size=2,
    res_size=params["res_size"],
    sparsity=params["sparsity"],
    spec_rad=params["spec_rad"],
    alpha=params["alpha"],
    input_scaling=params["input_scaling"],
    bias_scaling=params["bias_scaling"],
    ridge_alpha=params["ridge_alpha"],
    noise_val=params["noise_val"],
    tau_steps=1,
)
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
r_2, mse

(-0.010832371284946185, 0.471350228314025)

In [164]:
henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()
vis.plot_param_importances(
    study,
    target=lambda t: t.values[0],
    target_name="MSE",
    evaluator=PedAnovaImportanceEvaluator(target_quantile=0.1),
).show()

/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_73189/498646441.py:7: ExperimentalWarning: PedAnovaImportanceEvaluator is experimental (supported from v3.6.0). The interface can change in the future.
  evaluator=PedAnovaImportanceEvaluator(target_quantile=0.1),
/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_73189/498646441.py:3: UserWarning: PedAnovaImportanceEvaluator computes the importances of params to achieve low `target` values. If this is not what you want, please modify target, e.g., by multiplying the output by -1.
  vis.plot_param_importances(


# Bayesian MSE First Couple Term

In [165]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 10, 2000),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 0.1, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 0.1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )
    mse = root_mean_squared_error(Y_test[:10], Y_pred[:10])

    return mse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #99...

In [166]:
print(f"MSE={study.best_value}")
print(f"Params: {study.best_params}")

MSE=0.19165953924680632
Params: {'res_size': 1462, 'sparsity': 0.41262673959874685, 'spec_rad': 1.935294093762791, 'alpha': 0.773208355113615, 'input_scaling': 1.6780291283117919, 'bias_scaling': 0.6001709665416772, 'ridge_alpha': 2.146911868284677e-09, 'noise_val': 0.03172149503099127}


In [167]:
params = study.best_trials[0].params
Y_test, Y_pred = henon_closed(
    in_size=2,
    out_size=2,
    res_size=params["res_size"],
    sparsity=params["sparsity"],
    spec_rad=params["spec_rad"],
    alpha=params["alpha"],
    input_scaling=params["input_scaling"],
    bias_scaling=params["bias_scaling"],
    ridge_alpha=params["ridge_alpha"],
    noise_val=params["noise_val"],
    tau_steps=1,
)
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
r_2, mse

(-0.9456875802354971, 0.6702015273954139)

In [168]:
henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()
vis.plot_param_importances(
    study,
    target=lambda t: t.values[0],
    target_name="MSE",
    evaluator=PedAnovaImportanceEvaluator(target_quantile=0.1),
).show()

/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_73189/498646441.py:7: ExperimentalWarning: PedAnovaImportanceEvaluator is experimental (supported from v3.6.0). The interface can change in the future.
  evaluator=PedAnovaImportanceEvaluator(target_quantile=0.1),
/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_73189/498646441.py:3: UserWarning: PedAnovaImportanceEvaluator computes the importances of params to achieve low `target` values. If this is not what you want, please modify target, e.g., by multiplying the output by -1.
  vis.plot_param_importances(


# Bayesian Mean and Variance

In [188]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 10, 2000),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 0.1, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 0.1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )
    mse = root_mean_squared_error(Y_test[:10], Y_pred[:10])

    test_mean, test_var = np.mean(Y_test, axis=0), np.var(Y_test, axis=0)
    pred_mean, pred_var = np.mean(Y_pred, axis=0), np.var(Y_pred, axis=0)

    mean_diff = np.mean((test_mean - pred_mean) ** 2)
    var_diff = np.mean((test_var - pred_var) ** 2)

    return mse, mean_diff, var_diff


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #33...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.1056613439386126e-16.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


[Optuna] Processing Trial #94...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.6541340604800047e-16.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


[Optuna] Processing Trial #99...

In [189]:
best_trials = study.best_trials

print("Pareto Front trials:")
for trial in best_trials:
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")

Pareto Front trials:
Trial 14: [1.571990585681811, 71.64157859504748, 0.021823155110573787]
Params: {'res_size': 1276, 'sparsity': 0.11952964749915869, 'spec_rad': 1.9173558335058336, 'alpha': 0.6223045344701552, 'input_scaling': 0.8350403552301523, 'bias_scaling': 0.9736460398243153, 'ridge_alpha': 0.00014940593768389805, 'noise_val': 0.04511483699994567}
Trial 21: [2.2068147494372776, 3.0436576742288364, 0.102771072716376]
Params: {'res_size': 1512, 'sparsity': 0.44176833155704703, 'spec_rad': 1.7786762239645122, 'alpha': 0.631241709452898, 'input_scaling': 0.37466860612106284, 'bias_scaling': 0.6188228522955003, 'ridge_alpha': 7.005441670504042e-05, 'noise_val': 8.164926862864672e-06}
Trial 26: [0.5658701562816175, 23.835436087391923, 0.0564386565644921]
Params: {'res_size': 976, 'sparsity': 0.20107818563090069, 'spec_rad': 1.7039050289682312, 'alpha': 0.6871401028111008, 'input_scaling': 1.8876920767541707, 'bias_scaling': 0.8582604120291719, 'ridge_alpha': 6.837225418004359e-05, '

In [190]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=params["res_size"],
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 14: [1.571990585681811, 71.64157859504748, 0.021823155110573787]
Params: {'res_size': 1276, 'sparsity': 0.11952964749915869, 'spec_rad': 1.9173558335058336, 'alpha': 0.6223045344701552, 'input_scaling': 0.8350403552301523, 'bias_scaling': 0.9736460398243153, 'ridge_alpha': 0.00014940593768389805, 'noise_val': 0.04511483699994567}
R^2: -91.71118099149575, MSE: 5.103908110158213


Trial 21: [2.2068147494372776, 3.0436576742288364, 0.102771072716376]
Params: {'res_size': 1512, 'sparsity': 0.44176833155704703, 'spec_rad': 1.7786762239645122, 'alpha': 0.631241709452898, 'input_scaling': 0.37466860612106284, 'bias_scaling': 0.6188228522955003, 'ridge_alpha': 7.005441670504042e-05, 'noise_val': 8.164926862864672e-06}
R^2: -9.972830183734576, MSE: 1.2305960504236348


Trial 26: [0.5658701562816175, 23.835436087391923, 0.0564386565644921]
Params: {'res_size': 976, 'sparsity': 0.20107818563090069, 'spec_rad': 1.7039050289682312, 'alpha': 0.6871401028111008, 'input_scaling': 1.8876920767541707, 'bias_scaling': 0.8582604120291719, 'ridge_alpha': 6.837225418004359e-05, 'noise_val': 0.0037786956360424664}
R^2: -49.73059696960186, MSE: 3.783386159885814


Trial 46: [1.0330667669341758, 123.1180759924204, 0.0020497360963586124]
Params: {'res_size': 869, 'sparsity': 0.17394022081077237, 'spec_rad': 1.2806588394764438, 'alpha': 0.7159972236075256, 'input_scaling': 1.6012100794088695, 'bias_scaling': 0.3960726301007043, 'ridge_alpha': 2.895985669000731e-10, 'noise_val': 2.3384103587304148e-05}
R^2: -235.06207829510564, MSE: 8.082757244771011


Trial 64: [0.5824184542271045, 6.671533348301468, 0.024159623508755917]
Params: {'res_size': 724, 'sparsity': 0.2564211751027528, 'spec_rad': 1.768769388198022, 'alpha': 0.4376117317825957, 'input_scaling': 1.7695599300714093, 'bias_scaling': 0.1978534166149519, 'ridge_alpha': 1.3306439606310423e-08, 'noise_val': 0.06024480604415897}
R^2: -14.10352446095118, MSE: 2.069138045895511


Trial 71: [0.42745286683542305, 0.007262253679857731, 1.0538770052967321]
Params: {'res_size': 960, 'sparsity': 0.021776215362764997, 'spec_rad': 1.5831175832383444, 'alpha': 0.7926314023658497, 'input_scaling': 0.7676810533002749, 'bias_scaling': 0.44689234000561573, 'ridge_alpha': 1.5478402324807036e-08, 'noise_val': 0.002631604240667981}
R^2: -3.4304049721757695, MSE: 1.047039256470843


Trial 83: [0.35027786095097174, 0.25494232207618067, 0.1315198916533208]
Params: {'res_size': 116, 'sparsity': 0.44176833155704703, 'spec_rad': 1.897780603645853, 'alpha': 0.704880039469689, 'input_scaling': 0.77285537592079, 'bias_scaling': 0.24047137974086882, 'ridge_alpha': 1.2934506072878341e-05, 'noise_val': 1.2417327376337875e-06}
R^2: -1.3108601081772693, MSE: 0.6710745447022332


Trial 89: [3.495399945149412, 2.9069298963378296, 0.06160037220638197]
Params: {'res_size': 206, 'sparsity': 0.2564211751027528, 'spec_rad': 1.768769388198022, 'alpha': 0.4726482417085317, 'input_scaling': 1.7695599300714093, 'bias_scaling': 1.1601175834156203, 'ridge_alpha': 6.239828479356779e-08, 'noise_val': 5.3902204105325665e-05}
R^2: -8.135382072840978, MSE: 1.5275313393701873


In [ ]:
params = study.best_trials[0].params
Y_test, Y_pred = henon_closed(
    in_size=2,
    out_size=2,
    res_size=params["res_size"],
    sparsity=params["sparsity"],
    spec_rad=params["spec_rad"],
    alpha=params["alpha"],
    input_scaling=params["input_scaling"],
    bias_scaling=params["bias_scaling"],
    ridge_alpha=params["ridge_alpha"],
    noise_val=params["noise_val"],
    tau_steps=1,
)
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
r_2, mse

In [183]:
henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()
vis.plot_param_importances(
    study,
    target=None,
    target_name=["MSE", "Mean", "Variance"],
    evaluator=PedAnovaImportanceEvaluator(target_quantile=0.1),
).show()

/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_73189/3372461198.py:7: ExperimentalWarning: PedAnovaImportanceEvaluator is experimental (supported from v3.6.0). The interface can change in the future.
  evaluator=PedAnovaImportanceEvaluator(target_quantile=0.1),
/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_73189/3372461198.py:3: UserWarning: PedAnovaImportanceEvaluator computes the importances of params to achieve low `target` values. If this is not what you want, please modify target, e.g., by multiplying the output by -1.
  vis.plot_param_importances(


In [71]:
optuna.visualization.plot_pareto_front(study).show()

# Bayesian Lypanauv

In [13]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)
    # print(f"[Optuna] Processing Trial #{trial.number}.")

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 10, 2000),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-1, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
        trial=trial,
    )

    return lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon)


study = optuna.create_study(
    directions=["minimize"],
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[I 2026-06-26 17:39:53,323] A new study created in memory with name: no-name-b2833710-5089-4c12-9492-282efe9cc0b6


[Optuna] Processing Trial #199...

In [15]:
pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
complete_trials = [
    t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
]

print(f"Study statistics: ")
print(f"  Number of finished trials: {len(study.trials)}")
print(f"  Number of pruned trials: {len(pruned_trials)}")
print(f"  Number of complete trials: {len(complete_trials)}")

optuna.visualization.plot_intermediate_values(study).show()

Study statistics: 
  Number of finished trials: 200
  Number of pruned trials: 147
  Number of complete trials: 53


In [16]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=params["res_size"],
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 6: [0.22446884354706]
Params: {'res_size': 383, 'sparsity': 0.45430413642554673, 'spec_rad': 1.9153939614328432, 'alpha': 0.5991342085815959, 'input_scaling': 0.4525486413423376, 'bias_scaling': 0.8810149864304068, 'ridge_alpha': 6.321228679413582e-07, 'noise_val': 0.00011764002754518829}
R^2: -246.72728683196215, MSE: 7.680661901262172


# Bayesian Lypanauv and MSE

In [29]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 10, 2000),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-1, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
        trial=trial,
    )

    mse = root_mean_squared_error(Y_test, Y_pred)

    return lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon), mse


study = optuna.create_study(directions=["minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #199...

In [30]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=params["res_size"],
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 40: [0.00021287845465068825, 0.7011691210042993]
Params: {'res_size': 1096, 'sparsity': 0.4497424846778923, 'spec_rad': 1.8396265823029936, 'alpha': 0.531730705030633, 'input_scaling': 0.4675411225125029, 'bias_scaling': 0.2876883128812097, 'ridge_alpha': 5.0491904334298717e-11, 'noise_val': 0.0017996077893219465}
R^2: -1.1945917848232053, MSE: 0.7011691210042993


Trial 149: [0.00046274586799655856, 0.6977914493209827]
Params: {'res_size': 210, 'sparsity': 0.21190267460891093, 'spec_rad': 1.7334882864330723, 'alpha': 0.7469923497204646, 'input_scaling': 0.7227957725445934, 'bias_scaling': 0.34588583101024983, 'ridge_alpha': 2.0823493503670285e-13, 'noise_val': 3.260967582736116e-05}
R^2: -1.172796459543572, MSE: 0.6977914493209827


Trial 178: [0.002216949464113656, 0.6593431281874356]
Params: {'res_size': 784, 'sparsity': 0.21190267460891093, 'spec_rad': 1.9944562448202352, 'alpha': 0.786184571108373, 'input_scaling': 1.0252798305044817, 'bias_scaling': 0.34588583101024983, 'ridge_alpha': 4.848685724444552e-11, 'noise_val': 4.170899290921618e-06}
R^2: -1.074704095417048, MSE: 0.6600149119526767


Trial 179: [1.2956005033052192e-06, 0.7122110164994397]
Params: {'res_size': 210, 'sparsity': 0.21190267460891093, 'spec_rad': 1.7334882864330723, 'alpha': 0.786184571108373, 'input_scaling': 0.7227957725445934, 'bias_scaling': 0.29302145281795483, 'ridge_alpha': 2.0823493503670285e-13, 'noise_val': 3.260967582736116e-05}
R^2: -1.196878932158595, MSE: 0.7122110164994397


Trial 190: [0.007979808540122205, 0.49253417577935454]
Params: {'res_size': 1268, 'sparsity': 0.21190267460891093, 'spec_rad': 1.9433204076246775, 'alpha': 0.7469923497204646, 'input_scaling': 0.21673234934039667, 'bias_scaling': 0.34588583101024983, 'ridge_alpha': 2.408732904223691e-07, 'noise_val': 0.0007270738341537049}
R^2: -0.10871671147518669, MSE: 0.4914349386202021


# Bayesian NRSME

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 10, 2000),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-1, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
        trial=trial,
    )

    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return nrmse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #99...

In [ ]:
pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
complete_trials = [
    t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(f"Study statistics: ")
print(f"  Number of finished trials: {len(study.trials)}")
print(f"  Number of pruned trials: {len(pruned_trials)}")
print(f"  Number of complete trials: {len(complete_trials)}")

Study statistics: 
  Number of finished trials: 100
  Number of pruned trials: 64
  Number of complete trials: 36


In [ ]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=params["res_size"],
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 7: [1.8735281644357593]
Params: {'res_size': 556, 'sparsity': 0.09986572677752935, 'spec_rad': 1.973932639974864, 'alpha': 0.3211503214362415, 'input_scaling': 1.6556281578822465, 'bias_scaling': 0.15437735079567552, 'ridge_alpha': 7.090450436825761e-11, 'noise_val': 1.4473041447821855e-06}
R^2: -3.86267001162, MSE: 1.0123442967202394


# Bayesian NRSME First Couple Terms

In [39]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 10, 2000),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-1, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
        trial=trial,
    )

    rmse = root_mean_squared_error(Y_test[:30], Y_pred[:30])
    std = np.std(Y_test[:30])
    nrmse = rmse / std

    return nrmse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #99...

In [42]:
pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
complete_trials = [
    t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(f"Study statistics: ")
print(f"  Number of finished trials: {len(study.trials)}")
print(f"  Number of pruned trials: {len(pruned_trials)}")
print(f"  Number of complete trials: {len(complete_trials)}")

Study statistics: 
  Number of finished trials: 100
  Number of pruned trials: 64
  Number of complete trials: 36


In [43]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=params["res_size"],
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 6: [4.123458656565351]
Params: {'res_size': 678, 'sparsity': 0.4523479978303626, 'spec_rad': 1.4818030767541952, 'alpha': 0.7927491665295441, 'input_scaling': 1.6816414677909035, 'bias_scaling': 0.7149815472704155, 'ridge_alpha': 3.3198181370840756e-12, 'noise_val': 2.74579589468336e-05}
R^2: -105.00373854952167, MSE: 5.485734639406468


# Bayesian Hauss and Testing Reservoir Sizes

In [13]:
def normalize_2d(data):
    min_vals = data.min(axis=0)
    max_vals = data.max(axis=0)
    return (data - min_vals) / (max_vals - min_vals)

In [19]:
import time

def objective(trial):
    start_time = time.time()
    # print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 10, 1000),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-1, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
        trial=trial,
    )

    drift_penalty = np.sum(np.abs(Y_pred) > 2.0) * 10

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    end_time = time.time()
    duration = end_time - start_time

    print(f"Trial {trial.number} | Duration: {duration:.2f}s | Params: {trial.params}")

    return max(h_uv, h_vu) + drift_penalty


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=50, n_jobs=-1)

Trial 1 | Duration: 2.93s | Params: {'res_size': 28, 'sparsity': 0.3434406259169213, 'spec_rad': 0.36484525669088796, 'alpha': 0.5398367956956875, 'input_scaling': 0.5403949040996358, 'bias_scaling': 1.8333246001059742, 'ridge_alpha': 3.888299670995774e-06, 'noise_val': 3.830658526628604e-05}
Trial 4 | Duration: 3.06s | Params: {'res_size': 118, 'sparsity': 0.06411049963165781, 'spec_rad': 0.25519029457667153, 'alpha': 0.4263567390525666, 'input_scaling': 0.8021520443224575, 'bias_scaling': 0.943688091953008, 'ridge_alpha': 1.1177854278726854e-12, 'noise_val': 0.0041574622433710245}
Trial 6 | Duration: 3.02s | Params: {'res_size': 71, 'sparsity': 0.08157734147195476, 'spec_rad': 0.34569148797273086, 'alpha': 0.14519347190192708, 'input_scaling': 0.47082607677808125, 'bias_scaling': 1.1488540363571103, 'ridge_alpha': 0.0007014133319739003, 'noise_val': 0.015069114257391028}
Trial 2 | Duration: 3.08s | Params: {'res_size': 78, 'sparsity': 0.38634033572318816, 'spec_rad': 1.52690673157835

In [20]:
pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
complete_trials = [
    t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(f"Study statistics: ")
print(f"  Number of finished trials: {len(study.trials)}")
print(f"  Number of pruned trials: {len(pruned_trials)}")
print(f"  Number of complete trials: {len(complete_trials)}")

Study statistics: 
  Number of finished trials: 50
  Number of pruned trials: 32
  Number of complete trials: 18


In [25]:
vis.plot_optimization_history(study).show()

# See which parameters are actually influencing the result
vis.plot_param_importances(study).show()

fig = vis.plot_slice(study)
fig.show()

In [16]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=params["res_size"],
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 9: [1270.385019301551]
Params: {'res_size': 160, 'sparsity': 0.3190861752098193, 'spec_rad': 1.8362636869292444, 'alpha': 0.6052025803241995, 'input_scaling': 0.6814910660980249, 'bias_scaling': 0.7606111031737007, 'ridge_alpha': 4.598271778579796e-05, 'noise_val': 0.0002323039493026974}
R^2: -1.5234147158363378, MSE: 0.7508868219105105


In [28]:
for size in [50, 100, 200, 400, 600, 800, 1000, 2000]:
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    print(f"Reservoir Size: {size}")
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    drift_penalty = np.sum(np.abs(Y_pred) > 2.0) * 10
    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)
    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    print(f"  R^2: {r_2}, MSE: {mse}")
    print(f"  Hausdorff Distance: {max(h_uv, h_vu)}, Drift Penalty: {drift_penalty}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Reservoir Size: 50
  R^2: -845.217922141639, MSE: 15.092169483429728
  Hausdorff Distance: 0.403041131862286, Drift Penalty: 40030


Reservoir Size: 100
  R^2: -0.9063836930072429, MSE: 0.6004116225187227
  Hausdorff Distance: 0.4378984631887685, Drift Penalty: 10


Reservoir Size: 200
  R^2: -8.680964383857585, MSE: 1.4700188538969705
  Hausdorff Distance: 0.4561587716504008, Drift Penalty: 16890


Reservoir Size: 400
  R^2: -2249.589449525786, MSE: 24.6423919469704
  Hausdorff Distance: 0.472989647760962, Drift Penalty: 40000


Reservoir Size: 600
  R^2: -292.6470945107056, MSE: 9.103957880947391
  Hausdorff Distance: 0.5470334014345508, Drift Penalty: 40000


Reservoir Size: 800
  R^2: -928.5065606845906, MSE: 16.229085896176944
  Hausdorff Distance: 0.6652698381551264, Drift Penalty: 79980


Reservoir Size: 1000
  R^2: -842.947252552662, MSE: 15.346978660463925
  Hausdorff Distance: 0.5279791530641162, Drift Penalty: 50340


Reservoir Size: 2000
  R^2: -67.02056998812733, MSE: 4.097444155198233
  Hausdorff Distance: 0.3774810696285462, Drift Penalty: 40000


# Ideas

Image things like Hausdorff distance

fractal space like lyanpauv